In [1]:
import open3d as o3d
import numpy as np


In [2]:


#RADIUS = 1
#MAX_NN = 10

# Load point cloud data from .npy file
point_cloud = np.load("point_cloud.npy")
voxel_centers = np.load("voxel_centers.npy")


In [3]:
point_cloud.shape
voxel_centers.shape

(54740, 3)

In [4]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(point_cloud)

In [5]:
alpha = 3# Adjust depending on your point cloud density
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)

In [ ]:

# Color it differently
mesh.paint_uniform_color([0.1, 0.7, 1.0])  # light blue
pcd.paint_uniform_color([0.0, 0.0, 0.0])  # black points for clarity

o3d.visualization.draw_geometries(
    [pcd, mesh],
    window_name="Alpha shape reconstruction",
    mesh_show_back_face=True
)


In [8]:
voxels = voxel_centers

In [9]:
scene = o3d.t.geometry.RaycastingScene()
mesh_id = scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))

# === 3. Prepare voxel centers as query points ===
# voxels: (N,3) numpy array of voxel centers

# === 4. Cast rays in +X direction (can choose other directions too) ===
rays = np.concatenate([
    voxels.astype(np.float32),               # origins: voxel centers
    np.tile(np.array([[1,0,0]], dtype=np.float32), (voxels.shape[0],1))  # directions: +X
], axis=1)

# Convert to Open3D Tensor
rays_o3d = o3d.core.Tensor(rays, dtype=o3d.core.Dtype.Float32)

# === 5. Count intersections ===
counts = scene.count_intersections(rays_o3d).numpy()

# === 6. Apply even-odd rule to determine inside/outside ===
# (Strictly, odd counts mean inside under standard ray casting parity rule)
inside_mask = (counts % 2 == 1)

# === 7. Separate inside and outside voxel centers ===
voxels_inside = voxels[inside_mask]

In [12]:
np.sum(inside_mask)

np.int64(8355)

In [11]:
np.save("inside_mask.npy", inside_mask)